In [34]:
import torch
from model.model import EncoderDecoderDAG
from utils.data import TranslateDataset, collate_fn, process_data
from torch.utils.data import DataLoader
from utils.load_tokenizer import load_tokenizer
from torch.optim import Adam
from typing import Tuple
from utils.fix_probs import fix_probs, masking
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from utils.checkpoint import try_loading, epoch_resume, save_checkpoint
from utils.decoding import greedy_decoding, lookahead

In [35]:
tokenizer, vocab_size = load_tokenizer()

In [36]:
pad_idx = tokenizer.pad_token_id
eos_idx = tokenizer.eos_token_id

In [37]:
factor = 4
emb_size = 256
num_heads = 8
max_seq_len = 100
max_vertices = max_seq_len * factor

In [38]:
layers = [(2,2), 1, (1,1)]
out_layers = 1

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
device

device(type='cuda')

In [41]:
def create_model_fallback_fn() -> Tuple[EncoderDecoderDAG, Adam]:
    model = EncoderDecoderDAG(vocab_size, emb_size, num_heads, max_seq_len, max_vertices, layers, out_layers)
    model.to(device)
    optm = Adam(model.parameters(), lr=1e-3)
    return model, optm
    

In [42]:
checkpoint_dir = "./checkpoints"
checkpoint_name = "naivedag.pt"

In [43]:
model_class = EncoderDecoderDAG
optm_class = Adam

In [44]:
model, optm, losses, log_dir, tokens_seen = try_loading(checkpoint_dir, checkpoint_name, model_class, optm_class, device, create_model_fallback_fn)

Resuming, have seen 110,000 epochs and 40,573,858 tokens
Have 26704968 trainable parameters
Logging to runs/run_at_2023-12-24_13-34-45


In [45]:
en_test = "How are you"

In [46]:
encoded = tokenizer(en_test, return_tensors="pt").input_ids.to(device)
batch_size, l = encoded.shape
decoder_tokens = torch.arange(0, l * factor).unsqueeze(0).expand(batch_size, -1).to(device)
target_lens, vertex_lens, token_mask, vertex_mask = process_data(encoded, pad_idx, factor)
log_transition_probs, log_emission_probs = model(encoded, decoder_tokens, token_mask, vertex_mask)
mask = masking(log_transition_probs, vertex_lens)

In [47]:
log_transition_probs[0][2]

tensor([-32.7593, -30.4013, -35.8382, -31.2322, -34.2641, -35.1094, -38.1497,
        -37.0553, -32.4991, -31.6282, -36.7125, -21.8149, -20.8479, -32.3770,
        -28.2435,   0.0000], device='cuda:0', grad_fn=<SelectBackward0>)

In [48]:
#flog_transition_probs = log_transition_probs.masked_fill(mask !=0, float('-inf'))
flog_transition_probs = fix_probs(log_transition_probs, mask)
b1_transitions = flog_transition_probs[0]
b1_emissions = log_emission_probs[0]

In [49]:
b1_transitions[2]

tensor([    -inf,     -inf,     -inf, -31.2322, -34.2641, -35.1094, -38.1497,
        -37.0553, -32.4991, -31.6282, -36.7125, -21.8149, -20.8479, -32.3770,
        -28.2435,   0.0000], device='cuda:0', grad_fn=<SelectBackward0>)

In [50]:
torch.argmax(b1_transitions, dim=1)

tensor([ 7, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,  0],
       device='cuda:0')

In [51]:
torch.argmax(b1_emissions, dim=1)

tensor([65001,  1068,  9737,   146,   146, 26030,   146,  9737,  1068, 26030,
          146,     0,     0,  9737,     0,     0], device='cuda:0')

In [52]:
decoded = greedy_decoding(b1_transitions, b1_emissions, eos_idx)

In [53]:
tokenizer.decode(decoded)

'<s> 年轻</s>'

In [54]:
decoded2 = lookahead(b1_transitions, b1_emissions, eos_idx)

In [55]:
tokenizer.decode(decoded2)

'<s> 你</s>'